# 02 — First-class Functions, Lambdas, and Closures

In Python, functions are **objects**. You can pass them around, store them in lists, return them from other functions. That capability is what makes decorators (next notebook) possible.

## Functions are first-class objects

In [1]:
def shout(s):
    return s.upper() + "!"

def whisper(s):
    return s.lower() + "..."

# Store functions in a list
voices = [shout, whisper]
for v in voices:
    print(v("Hello there"))

# Pass a function as an argument
def transform_all(words, func):
    return [func(w) for w in words]

print(transform_all(["foo", "bar"], shout))

# Functions have attributes
print(shout.__name__, shout.__doc__)

HELLO THERE!
hello there...
['FOO!', 'BAR!']
shout None


## Lambdas — tiny anonymous functions

`lambda params: expression` is shorthand for a single-expression function. Don't use them for anything you'd otherwise name — they show up most often as `key=` arguments to `sorted` / `max` / `min`.

In [2]:
add = lambda a, b: a + b   # equivalent to:  def add(a, b): return a + b
print(add(2, 3))

# The natural home of lambda: sort/min/max keys
people = [("alice", 90), ("bob", 75), ("cleo", 60)]
print(sorted(people, key=lambda p: p[1]))             # by score
print(sorted(people, key=lambda p: p[1], reverse=True))
print(max(people, key=lambda p: p[1]))

# `operator.itemgetter` is cleaner for this case:
from operator import itemgetter
print(sorted(people, key=itemgetter(1)))

5
[('cleo', 60), ('bob', 75), ('alice', 90)]
[('alice', 90), ('bob', 75), ('cleo', 60)]
('alice', 90)
[('cleo', 60), ('bob', 75), ('alice', 90)]


**Don't** write `f = lambda x: x*2`. Use `def`. Lambdas are for *in-line, one-shot* use.

## Closures — functions that remember

When you define a function *inside* another function, the inner function can read variables from the outer function's scope. The inner function plus that captured environment is a **closure**.

In [3]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor       # `factor` is captured from the enclosing scope
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10))   # 20
print(triple(10))   # 30

# Each closure has its own `factor`. They don't interfere.
print(double.__closure__)         # tuple of cells holding the captured vars
print(double.__closure__[0].cell_contents)

20
30
(<cell at 0x000001095EDAB400: int object at 0x00007FFFF034E9D8>,)
2


## `nonlocal` — when the inner function needs to *write* to the captured variable

Reading from the enclosing scope is free. *Reassigning* needs `nonlocal` (for an enclosing function's variable) or `global` (for a module-level variable). Without it, Python treats the name as a new local.

In [4]:
def make_counter(start=0):
    count = start
    def step():
        nonlocal count        # without this, `count = count + 1` would create a NEW local count
        count += 1
        return count
    return step

next_val = make_counter()
print(next_val(), next_val(), next_val())   # 1 2 3

# Each closure is independent:
a = make_counter(100)
b = make_counter(100)
print(a(), a(), b(), a(), b())

1 2 3
101 102 101 103 102


## The classic late-binding trap

Closures capture variables **by reference**, not by value. If you build closures in a loop and the loop variable changes, *all* the closures see the final value.

This is the bug behind exercise #5 in the README.

In [5]:
# BUG: all closures see i == 2 at call time
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])     # [2, 2, 2]

# FIX A: turn `i` into a default argument — defaults are evaluated when def is executed
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])     # [0, 1, 2]

# FIX B: factor out a function that takes i as a parameter
def make_returner(i):
    return lambda: i
funcs = [make_returner(i) for i in range(3)]
print([f() for f in funcs])     # [0, 1, 2]

[2, 2, 2]
[0, 1, 2]
[0, 1, 2]


## A useful closure: an accumulator that hides its state

Closures give you *encapsulation without a class*. Sometimes that's exactly what you want; sometimes a class is clearer. We'll see both.

In [6]:
def make_accumulator():
    total = 0
    def add(x):
        nonlocal total
        total += x
        return total
    return add

acc = make_accumulator()
print(acc(10))   # 10
print(acc(5))    # 15
print(acc(-3))   # 12
# There is no way to inspect or reset `total` from the outside — it's hidden.

10
15
12


## Mini-exercise

1. Write `compose(f, g)` that returns a new function `h` such that `h(x) == f(g(x))`. Test: `compose(str.upper, str.strip)("  hello  ") == "HELLO"`.
2. Write `once(func)` that wraps `func` so it runs only the first call; subsequent calls return the cached first result.
3. Predict, then run:
   ```python
   adders = []
   for n in [1, 10, 100]:
       adders.append(lambda x: x + n)
   print([a(0) for a in adders])
   ```
   Then fix it two different ways.

In [7]:
# your solutions here


**Takeaways**

- Functions are objects: pass them, store them, return them.
- Lambdas are for short one-shot expressions, typically as `key=` arguments.
- A **closure** = inner function + captured variables from its enclosing scope.
- `nonlocal` lets the inner function *reassign* an enclosing variable.
- Closures capture variables by **reference** — the late-binding trap is real.

Next: [03_decorators.ipynb](03_decorators.ipynb) — closures, applied.